In [2]:
import os
from dotenv import load_dotenv
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

load_dotenv(override=True)

# 임베딩 객체 선언
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
#embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
#query_result = embeddings.embed_query("저는 배가 불러요")
query_result = embeddings.embed_query("저는 배가 고파요")
print("벡터 차원 수:", len(query_result))  #1536 차원
print("임베딩 결과 (일부):", query_result[:5])

# 1. 7개의 샘플 문장 코퍼스 정의
data = [
    "저는 배가 고파요",
    "저는 배가 엄청 불러요",
    "저기 배가 지나가네요",
    "굶어서 허기가 지네요",
    "허기 위기라는 게임이 있는데 즐거워",
    "스팀에서 재밌는 거 해야지",
    "스팀에어프라이어에 연어구이 해먹을거야"
]

df = pd.DataFrame(data, columns=["text"])

# 2. 텍스트 임베딩 변환 함수 정의
def get_embedding(text: str):
    return embeddings.embed_query(text)
    

# 3. 데이터프레임 전체에 적용하여 embedding 컬럼 생성
df["embedding"] = df.apply(lambda row: get_embedding(row["text"]), axis=1)
print(df)

def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

def return_answer_candidate(df: pd.DataFrame, query: str):

    # 1. 사용자 입력 질의 벡터화
    query_embedding = get_embedding(query)
    
    # 2. 문서 집합 내 모든 벡터와 코사인 유사도 일괄 연산
    df["similarity"] = df["embedding"].apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))
    
    # 3. 유사도가 높은 순(내림차순)으로 상위 3개 문서 추출
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)
    return top_three_doc

# 의미 검색 테스트: 직접적인 키워드가 없는 문장 질의
sim_result = return_answer_candidate(df, "아무것도 못먹었더니 배가 꼬르륵 거리네요")
print(sim_result[["text", "similarity"]])



벡터 차원 수: 1536
임베딩 결과 (일부): [0.017364501953125, -0.0078582763671875, -0.0290985107421875, 0.0075531005859375, 0.00115203857421875]
                   text                                          embedding
0             저는 배가 고파요  [0.017364501953125, -0.0078582763671875, -0.02...
1          저는 배가 엄청 불러요  [0.05108642578125, -0.0077362060546875, -0.063...
2           저기 배가 지나가네요  [0.03948974609375, 8.654594421386719e-05, -0.0...
3           굶어서 허기가 지네요  [0.041534423828125, 0.0248260498046875, -0.012...
4   허기 위기라는 게임이 있는데 즐거워  [0.0171051025390625, -0.0201416015625, -0.0440...
5        스팀에서 재밌는 거 해야지  [0.01322174072265625, 0.05426025390625, -0.040...
6  스팀에어프라이어에 연어구이 해먹을거야  [0.043548583984375, 0.0012388229370117188, 0.0...
           text  similarity
1  저는 배가 엄청 불러요    0.442392
0     저는 배가 고파요    0.331290
2   저기 배가 지나가네요    0.315439
